# 2b — ICI Specificity of CIDER T-cell Score: Overall Survival

**Purpose**: Builds Panel A of the combined `supplement_survival_specificity.png`
(assembled by `assemble_survival_specificity_figure.R`, alongside Panel B from
`3b_survival_tmb_specificity.ipynb`). Supplementary to Figure 4
(`2_survival.ipynb`): Figure 4 shows that CIDER T-cell High vs Low
stratifies overall survival within ICI-treated patients, faceted by cancer
type. This panel asks the specificity question instead: does that same
CIDER High/Low split predict survival *only* in ICI therapy, or does it
also show an effect in patients who never received ICI?

**Cohort definitions** are identical to `15_ici_specificity.ipynb`'s
`parent_dfs` (ICI Therapy, Non-ICI Therapy, Chemotherapy, Targeted therapy,
Multiple therapy, Hormonal therapy) -- see that notebook's intro for the
detailed inclusion/exclusion logic for each treatment class.

**CIDER High/Low threshold**: a single cutoff is reused across every panel,
rather than an independent median split per panel. The cutoff is Figure 4's
own ICI-treated split (`cpi_ready.rds`'s `group` column: a median split of
the depth-normalized CIDER T-cell Score within the ICI-treated cohort,
computed in `0_prepare/4_base.ipynb`). "ICI Therapy" reuses that column
directly, so it is identical to Figure 4's own ICI-treated classification.
Every other panel classifies its own patients as High/Low relative to that
same numeric cutoff (the maximum score in Figure 4's "Low" bucket -- patients
scoring above it are "High", at or below are "Low"). This tests whether a
CIDER score that is "High" by the ICI cohort's own definition also predicts
survival outside ICI therapy; re-splitting each panel at its own internal
median would instead force every panel into an artificial 50/50 split and
couldn't support that comparison.

Note: `cpi_ready.rds` carries two rows per ICI-treated patient (their natural
cancer-type cohort_broad, plus a duplicate with cohort_broad forced to
"ICI-Treated" -- see `4_base.ipynb`). The "ICI Therapy" panel's threshold
lookup deduplicates by sampleId before joining, to avoid double-counting
those patients.

**Panels (6 total, 3x2 layout):** ICI Therapy, Non-ICI Therapy,
Chemotherapy, Targeted therapy, Hormonal therapy, Multiple therapy.

Each panel annotates the Cox HR (CIDER High vs Low) and log-rank p-value, as
in Figure 4. No FDR correction is applied here (6 descriptive KM panels, not
a formal hypothesis-testing forest plot like `15_ici_specificity.ipynb`) --
read the HRs/p-values as supporting visual evidence alongside that notebook's
formal interaction tests, not as a replacement for them.

| | |
|---|---|
| **Inputs** | `CIDER_PROJ_DIR/ready.rds`, `cpi_ready.rds`, `clinical_update.txt` |
| **Outputs** | `figs/survival_specificity_cider_panel.rds` (Panel A intermediate -- combined into `figs/supplement_survival_specificity.png` by `assemble_survival_specificity_figure.R`) |


In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))   # ICI_TREATMENT_NORM, ICI_MECHANISM_MAP
source(paste0(dirname(getwd()), "/helpers/themes.r"))   # go_theme, high_low_color, figure_annotation, fmt_pval


In [ ]:
library(survival)
library(survminer)
library(patchwork)


## 1 — Data preparation


In [ ]:
# Load clinical metadata. Identical construction to 15_ici_specificity.ipynb,
# so treatment-type/mechanism classification matches that notebook exactly.
# 'treatment_str' is renamed from 'treatment' to avoid collision with the
# existing 'treatment' column in ready.rds after the join.
clinical_raw <- data.table::fread(paste0(READY_DIR, "clinical_update.txt")) |>
  dplyr::transmute(
    sampleId       = sampleId,
    treatment_type = type,
    treatment_str  = treatment,
    mechanism_raw  = mechanism
  ) |>
  dplyr::distinct(sampleId, .keep_all = TRUE)


In [ ]:
# Load full ready object (T-cell group); join clinical metadata.
# ready.rds duplicates every patient: once with the natural cohort_broad and
# once with cohort_broad = 'ICI-Treated' (see 4_base.ipynb bind_rows call).
# Deduplicate by sampleId immediately to avoid inflated Ns.
# Classify non-ICI treatment regimens into mechanism sub-groups based on
# the treatment string. Returns NA for groups too small to be informative
# (e.g. irinotecan alone, ICI-containing combinations, anti-EGFR).
classify_non_ici_mech <- function(trt) {
  if (is.na(trt) || trt == "") return(NA_character_)
  t <- tolower(trt)
  has <- function(...) any(vapply(c(...), function(d) grepl(d, t, fixed = TRUE), logical(1)))
  # Exclude ICI-containing regimens from non-ICI sections
  if (has("pembrolizumab","nivolumab","atezolizumab","ipilimumab",
           "durvalumab","avelumab","tremelimumab")) return(NA_character_)
  # Targeted / biologic add-ons
  if (has("palbociclib","ribociclib","abemaciclib"))              return("CDK4/6i")
  if (has("trastuzumab","pertuzumab","lapatinib","ado-trastuzumab")) return("Chemo + Anti-HER2")
  if (has("bevacizumab","ramucirumab"))                           return("Chemo + Anti-VEGF")
  # Cetuximab/panitumumab: too few responders to model — drop
  if (has("cetuximab","panitumumab"))                             return(NA_character_)
  # Pure chemotherapy sub-groups (ordered from most specific to most general)
  has_plat <- grepl("carboplatin|cisplatin|oxaliplatin", t)
  has_gem  <- grepl("gemcitabine", t)
  has_tax  <- grepl("paclitaxel|docetaxel|cabazitaxel|nab-paclitaxel", t)
  has_fp   <- grepl("capecitabine|fluorouracil|trifluridine|tipiracil", t)
  has_anth <- grepl("doxorubicin|epirubicin", t)
  has_iri  <- grepl("irinotecan", t)
  has_etop <- grepl("etoposide", t)
  if (has_gem  && has_plat && !has_tax && !has_fp)  return("Gemcitabine + Platinum")
  if (has_tax  && has_plat)                         return("Taxane + Platinum")
  if (has_fp   && grepl("oxaliplatin", t) && !has_tax) return("Fluoropyrimidine + Oxaliplatin")
  # Irinotecan alone or with FP: too few responders — drop
  if (has_iri)                                      return(NA_character_)
  if (has_tax  && !has_plat)                        return("Taxane")
  if (has_fp   && !has_plat && !has_anth)           return("Fluoropyrimidine")
  if (has_anth)                                     return("Anthracycline")
  if (has_etop && has_plat)                         return("Etoposide + Platinum")
  if (has_plat && !has_gem && !has_tax && !has_fp)  return("Platinum")
  return(NA_character_)
}

ready <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds")) |>
  dplyr::filter(gp == "T-cell") |>
  dplyr::distinct(sampleId, .keep_all = TRUE) |>
  dplyr::left_join(clinical_raw, by = "sampleId") |>
  dplyr::mutate(
    cider_log2   = log2(cider_score_depth_adj + 1),
    response   = as.integer(response),
    dcb        = as.integer(dcb),
    os_months  = osDays / 30.44,
    pfs_months = pfsEventDays / 30.44,
    cohort_pan = as.character(cohort_pan)
  ) |>
  dplyr::mutate(
    treatment_norm = unname(ICI_TREATMENT_NORM[treatment_str]),
    treatment_norm = dplyr::coalesce(treatment_norm, treatment_str)
  ) |>
  dplyr::mutate(
    ici_mechanism   = unname(ICI_MECHANISM_MAP[treatment_norm]),
    ici_mechanism   = dplyr::coalesce(ici_mechanism, "Other"),
    # ICI patients: use ICI mechanism map.
    # Chemo/Multiple therapy: reclassify by treatment string.
    # Targeted/Hormonal therapy: keep original mechanism_raw label.
    mechanism_group = dplyr::case_when(
      cpi == TRUE ~ ici_mechanism,
      treatment_type %in% c("Chemotherapy", "Multiple therapy") ~
        unname(vapply(treatment_str, classify_non_ici_mech, character(1))),
      TRUE ~ mechanism_raw
    )
  ) |>
  dplyr::select(-treatment_norm, -ici_mechanism) |>
  # Restrict to patients with both response (BOR) and OS annotation,
  # matching the inclusion criteria used in the main analyses.
  dplyr::filter(!is.na(response), !is.na(os_months), !is.na(osEvent))

message("Total unique patients: ", dplyr::n_distinct(ready$sampleId))


## 2 — Cohort definitions

Same treatment-class groups as `15_ici_specificity.ipynb`'s `parent_dfs`.


In [ ]:
# Top-level treatment-class groups -- identical definitions to
# 15_ici_specificity.ipynb's parent_dfs, so cohort membership is consistent
# across the two supplementary figures.
parent_dfs <- list(
  "ICI Therapy"      = ready |> dplyr::filter(cpi == TRUE) |>
                        dplyr::mutate(cohort_pan = as.character(cohort_broad)),
  "Non-ICI Therapy"  = ready |> dplyr::filter(cpi == FALSE),
  "Chemotherapy"     = ready |> dplyr::filter(cpi == FALSE, treatment_type == "Chemotherapy"),
  "Targeted therapy" = ready |> dplyr::filter(cpi == FALSE,
                          treatment_type == "Targeted therapy" | mechanism_group == "CDK4/6i"),
  "Multiple therapy" = ready |> dplyr::filter(cpi == FALSE,
                          mechanism_group %in% c("Chemo + Anti-HER2", "Chemo + Anti-VEGF")),
  "Hormonal therapy" = ready |> dplyr::filter(cpi == FALSE, treatment_type == "Hormonal therapy")
)

for (nm in names(parent_dfs)) {
  message(nm, ": n = ", nrow(parent_dfs[[nm]]), ", events (OS) = ", sum(parent_dfs[[nm]]$osEvent, na.rm = TRUE))
}


## 3 — CIDER High/Low threshold (reused from Figure 4)


In [ ]:
# Figure 4's own ICI-treated CIDER High/Low split (0_prepare/4_base.ipynb:
# ntile(cider_score_depth_adj, 2), grouped by gp only -- i.e. one threshold
# across the whole ICI-treated cohort, not per cancer type). Reused here as a
# single fixed cutoff for every panel below.
# ready.rds (and cpi_ready.rds, derived from it) duplicates every
# ICI-treated patient: once under their natural cohort_broad (e.g. "SKCM")
# and once with cohort_broad forced to "ICI-Treated" (4_base.ipynb's
# bind_rows call, for Figure 3/4's pooled-vs-per-cancer-type panels). Both
# copies have cpi == TRUE, so cpi_ready.rds carries two rows per ICI-treated
# patient for gp == "T-cell". Deduplicate by sampleId here -- group and
# cider_score_depth_adj are identical between the two copies -- otherwise
# joining against this below would double-count every ICI Therapy patient.
cpi_ready_group <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  dplyr::filter(gp == "T-cell") |>
  dplyr::distinct(sampleId, .keep_all = TRUE) |>
  dplyr::select(sampleId, group, cider_score_depth_adj)

# The effective ntile(2) boundary: the largest depth-normalized CIDER score
# among Figure 4's "Low" patients. A score above this value is "High", at or
# below is "Low" -- applied identically to every non-ICI panel below.
cider_cutoff <- max(
  cpi_ready_group$cider_score_depth_adj[cpi_ready_group$group == "Low"],
  na.rm = TRUE
)
message("CIDER High/Low cutoff (Figure 4's ICI-treated median): ", round(cider_cutoff, 3))

# Classify High/Low relative to the shared cutoff above, for any population
# that isn't already covered by Figure 4's own `group` column.
assign_group_from_cutoff <- function(df) {
  df |>
    dplyr::mutate(
      group = factor(
        ifelse(cider_score_depth_adj > cider_cutoff, "High", "Low"),
        levels = c("Low", "High")
      )
    )
}

# ICI Therapy: reuse Figure 4's own group column directly (exact match, not
# a recomputation) via a join on sampleId.
join_ici_group <- function(df) {
  n_before <- nrow(df)
  df <- df |>
    dplyr::select(-dplyr::any_of("group")) |>
    dplyr::inner_join(cpi_ready_group |> dplyr::select(sampleId, group), by = "sampleId")
  if (nrow(df) != n_before) {
    message("  (n changed ", n_before, " -> ", nrow(df), " after joining cpi_ready.rds group -- check for duplicates)")
  }
  df
}

panel_dfs <- list(
  "ICI Therapy"      = join_ici_group(parent_dfs[["ICI Therapy"]]),
  "Non-ICI Therapy"  = parent_dfs[["Non-ICI Therapy"]] |> assign_group_from_cutoff(),
  "Chemotherapy"     = parent_dfs[["Chemotherapy"]]    |> assign_group_from_cutoff(),
  "Targeted therapy" = parent_dfs[["Targeted therapy"]] |> assign_group_from_cutoff(),
  "Hormonal therapy" = parent_dfs[["Hormonal therapy"]] |> assign_group_from_cutoff(),
  "Multiple therapy" = parent_dfs[["Multiple therapy"]] |> assign_group_from_cutoff()
)

for (nm in names(panel_dfs)) {
  message(nm, ": ", paste(capture.output(print(table(panel_dfs[[nm]]$group))), collapse = " | "))
}


## 4 — KM panel function


In [ ]:
MIN_N      <- 20
MIN_EVENTS <- 5

# One KM panel per named entry in `dfs`: CIDER High/Low curves for overall
# survival, with Cox HR + log-rank p annotated (same style as
# 2_survival.ipynb's survival_plot()). Returns NULL (and skips) if the panel
# doesn't clear MIN_N/MIN_EVENTS or coxph() fails to fit.
km_specificity_panel <- function(df, panel_label) {
  df <- df |> tidyr::drop_na(group, os_months, osEvent) |> dplyr::filter(os_months > 0)

  if (nrow(df) < MIN_N || sum(df$osEvent, na.rm = TRUE) < MIN_EVENTS ||
      dplyr::n_distinct(df$group) < 2) {
    message("Skipping ", panel_label, " (n=", nrow(df), ", events=", sum(df$osEvent, na.rm = TRUE), ")")
    return(NULL)
  }

  fit <- tryCatch(survival::survfit(Surv(os_months, osEvent) ~ group, data = df), error = function(e) NULL)
  cox <- tryCatch(survival::coxph(Surv(os_months, osEvent) ~ group, data = df), error = function(e) NULL)
  if (is.null(fit) || is.null(cox)) {
    message("Skipping ", panel_label, " (model fit failed)")
    return(NULL)
  }

  clean_labels <- gsub("group=", "", names(fit$strata))
  hr      <- round(exp(coef(cox)), 2)
  p_val   <- summary(cox)$logtest[3]
  p_label <- fmt_pval(p_val)

  # Median OS per CIDER group (Low/High) -- same survminer::surv_median()
  # convention as 3_survival_with_tmb.ipynb's pan_cancer_survival_tmb (Figure
  # 5C) and 2_survival.ipynb's survival_plot() (Figure 4). "NR" (not reached)
  # when a stratum's KM curve never crosses 0.5 survival.
  med_surv <- survminer::surv_median(fit) |>
    dplyr::mutate(group_clean = gsub("group=", "", strata))
  med_label <- function(grp) {
    row <- med_surv |> dplyr::filter(group_clean == grp)
    if (nrow(row) == 0) return(NULL)
    if (is.na(row$median)) paste0(grp, ": NR") else sprintf("%s: %.1f mo", grp, row$median)
  }
  median_lines <- paste(c("Median OS:", med_label("Low"), med_label("High")), collapse = "\n")

  annot   <- paste0("HR: ", hr, "\n", p_label, "\nN = ", nrow(df), "\n", median_lines)

  ggsurvplot(
    fit, data = df,
    legend.labs = clean_labels,
    pval        = annot,
    # Shifted left (0.55 -> 0.45) to leave more room for the now-taller
    # annotation block (HR/p-value/N + the added Median OS lines) to grow
    # rightward without running past the panel's right edge. y lowered
    # (0.95 -> 0.85) too -- the block is now 6 lines tall (HR, p-value, N,
    # "Median OS:", Low, High), and anchored at 0.95 it ran past the top of
    # the survival-probability axis (pushing the p-value line above the
    # plot).
    pval.coord  = c(max(df$os_months, na.rm = TRUE) * 0.55, 0.75),
    pval.size   = 3,
    title       = panel_label
  )$plot +
    go_theme +
    theme(plot.title = element_text(hjust = 0.5)) +
    scale_color_manual(values = high_low_color) +
    guides(color = guide_legend(ncol = 1)) +
    labs(color = "CIDER T-cell Score", x = "Time (months)")
}


## 5 — Assemble figure


In [ ]:
PANEL_ORDER <- c(
  "ICI Therapy", "Non-ICI Therapy",
  "Chemotherapy", "Targeted therapy",
  "Hormonal therapy", "Multiple therapy"
)

panels <- purrr::imap(panel_dfs[PANEL_ORDER], km_specificity_panel)
panels <- purrr::compact(panels)  # drop any panel that failed MIN_N/MIN_EVENTS or model fit

if (length(panels) < length(PANEL_ORDER)) {
  warning(length(PANEL_ORDER) - length(panels), " panel(s) skipped -- see messages above.")
}

# Intermediate panel only -- no standalone .png here. Combined with
# 3b_survival_tmb_specificity.ipynb's panel into the final
# supplement_survival_specificity.png by assemble_survival_specificity_figure.R
# (Panel A = this grid, Panel B = the CIDER x TMB grid). No collected legend
# is shown -- the High/Low colour mapping is the same high_low_color scheme
# used throughout the paper, and each sub-panel is already labelled/
# annotated (HR, p-value, N), so a repeated legend box was redundant.
survival_specificity_cider_panel <-
  patchwork::wrap_plots(panels, ncol = 3, guides = "collect") +
  figure_annotation(
    "CIDER T-cell Score and Overall Survival: Specificity to ICI Therapy",
    tag = FALSE, legend_pos = "none"
  )

set_plot_dims(12, 7.5); survival_specificity_cider_panel

saveRDS(survival_specificity_cider_panel, paste0(CIDER_PROJ_DIR, "figs/survival_specificity_cider_panel.rds"))
message("Saved survival_specificity_cider_panel.rds (Panel A intermediate)")
